In [1]:
from pathlib import Path
from zipfile import ZipFile
import io
import numpy as np
import pandas as pd


DATA_ROOT = Path("../../hcris-data/HCRIS_v2010").resolve()

OUTPUT_DIR = Path("output").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



HCRIS_VARS = [
    # (name, wksht_cd, line_num, clmn_num, source)
    ("beds",                        "S300001", "01400", "00200", "numeric"),
    ("tot_charges",                "G300000", "00100", "00100", "numeric"),
    ("tot_discounts",              "G300000", "00200", "00100", "numeric"),
    ("net_pat_rev",                "G300000", "00300", "00100", "numeric"),
    ("tot_operating_exp",          "G300000", "00400", "00100", "numeric"),
    ("ip_charges",                 "G200000", "00100", "00100", "numeric"),
    ("icu_charges",                "G200000", "01600", "00100", "numeric"),
    ("ancillary_charges",          "G200000", "01800", "00100", "numeric"),
    ("tot_discharges",             "S300001", "00100", "01500", "numeric"),
    ("mcare_discharges",           "S300001", "00100", "01300", "numeric"),
    ("mcaid_discharges",           "S300001", "00100", "01400", "numeric"),
    ("tot_mcare_payment",          "E00A18A", "05900", "00100", "numeric"),
    ("secondary_mcare_payment",    "E00A18A", "06000", "00100", "numeric"),
    ("street",                     "S200001", "00100", "00100", "alpha"),
    ("city",                       "S200001", "00200", "00100", "alpha"),
    ("state",                      "S200001", "00200", "00200", "alpha"),
    ("zip",                        "S200001", "00200", "00300", "alpha"),
    ("county",                     "S200001", "00200", "00400", "alpha"),
    ("name",                       "S200001", "00300", "00100", "alpha"),
    ("hvbp_payment",               "E00A18A", "07093", "00100", "numeric"),
    ("hrrp_payment",               "E00A18A", "07094", "00100", "numeric"),
    ("tot_uncomp_care_charges",    "S100000", "02000", "00300", "numeric"),
    ("tot_uncomp_care_partial_pmts","S100000", "02200", "00300", "numeric"),
    ("bad_debt",                   "S100000", "02800", "00100", "numeric"),
    ("cost_to_charge",             "S100000", "00100", "00100", "numeric"),
    ("new_cap_ass",                "A700001", "01000", "00200", "numeric"),
    ("cash",                       "G000000", "00100", "00100", "numeric"),
    ("fixed_assets",               "G000000", "03000", "00100", "numeric"),
    ("depr_land",                  "G000000", "01400", "00100", "numeric"),
    ("depr_bldg",                  "G000000", "01600", "00100", "numeric"),
    ("depr_lease",                 "G000000", "01800", "00100", "numeric"),
    ("depr_fixed_equip",           "G000000", "02000", "00100", "numeric"),
    ("depr_auto",                  "G000000", "02200", "00100", "numeric"),
    ("depr_major_equip",           "G000000", "02400", "00100", "numeric"),
    ("depr_minor_equip",           "G000000", "02600", "00100", "numeric"),
    ("depr_HIT",                   "G000000", "02800", "00100", "numeric"),
    ("current_assets",             "G000000", "01100", "00100", "numeric"),
    ("current_liabilities",        "G000000", "04500", "00100", "numeric"),
]

RPT_COLS = [
    "RPT_REC_NUM", "PRVDR_CTRL_TYPE_CD", "PRVDR_NUM", "NPI",
    "RPT_STUS_CD", "FY_BGN_DT", "FY_END_DT", "PROC_DT",
    "INITL_RPT_SW", "LAST_RPT_SW", "TRNSMTL_NUM", "FI_NUM",
    "ADR_VNDR_CD", "FI_CREAT_DT", "UTIL_CD", "NPR_DT",
    "SPEC_IND", "FI_RCPT_DT",
]

LONG_COLS = ["RPT_REC_NUM", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "ITM_VAL_NUM"]



def _resolve_path(year, suffix):
    folders = [f"HospitalFY{year}", f"HOSP10FY{year}"]
    files = [
        f"hosp10_{year}_{suffix.upper()}.CSV",
        f"HOSP10_{year}_{suffix.lower()}.csv",
    ]
    for folder in folders:
        for fname in files:
            path = DATA_ROOT / folder / fname
            if path.exists():
                return path, str(path)

    # ZIP fallback
    zips = [
        f"HOSP10FY{year}.ZIP", f"HOSP10FY{year}.zip",
        f"HospitalFY{year}.ZIP", f"HospitalFY{year}.zip",
    ]
    entries = [
        f"HOSP10_{year}_{suffix.lower()}.csv",
        f"hosp10_{year}_{suffix.upper()}.CSV",
    ]
    for zf in zips:
        zip_path = DATA_ROOT / zf
        if zip_path.exists():
            with ZipFile(zip_path) as zfile:
                names = zfile.namelist()
                for entry in entries:
                    if entry in names:
                        buf = io.BytesIO(zfile.read(entry))
                        return buf, f"{zip_path}:{entry}"

    raise FileNotFoundError(f"Cannot find HCRIS v2010 data for year {year}, suffix {suffix}")


def _read_csv(year, suffix, col_names):
    source, label = _resolve_path(year, suffix)
    return pd.read_csv(source, header=None, names=col_names, dtype=str)



def extract_v2010():
    print(f"Extracting HCRIS v2010 (2010-2025) from: {DATA_ROOT}")
    frames = []
    for yr in range(2010, 2026):
        try:
            alpha = _read_csv(yr, "alpha", LONG_COLS)
            nmrc = _read_csv(yr, "nmrc", LONG_COLS)
            rpt = _read_csv(yr, "rpt", RPT_COLS)

            final = rpt[["RPT_REC_NUM", "PRVDR_NUM", "NPI", "FY_BGN_DT", "FY_END_DT",
                          "PROC_DT", "FI_CREAT_DT", "RPT_STUS_CD"]].copy()
            final.columns = ["report", "provider_number", "npi", "fy_start", "fy_end",
                             "date_processed", "date_created", "status"]
            final["year"] = yr
            final["data_source"] = "v2010"

            for var_name, wksht, line, col, source in HCRIS_VARS:
                data = alpha if source == "alpha" else nmrc
                vals = (
                    data
                    .loc[(data["WKSHT_CD"] == wksht) &
                         (data["LINE_NUM"] == line) &
                         (data["CLMN_NUM"] == col),
                         ["RPT_REC_NUM", "ITM_VAL_NUM"]]
                    .rename(columns={"RPT_REC_NUM": "report", "ITM_VAL_NUM": var_name})
                )
                
                # Ensure 'report' column is string type for a clean merge
                vals["report"] = vals["report"].astype(str)
                final["report"] = final["report"].astype(str)

                if source == "numeric":
                    vals[var_name] = pd.to_numeric(vals[var_name], errors="coerce")
                final = final.merge(vals, on="report", how="left")

            frames.append(final)
            print(f"  v2010 {yr}: {len(final):,} reports")
            
        except FileNotFoundError:
            print(f"  v2010 {yr}: Files not found (Skipping)")
            continue

    if frames:
        combined = pd.concat(frames, ignore_index=True)
        output_path = OUTPUT_DIR / "HCRIS_Data_v2010.txt"
        combined.to_csv(output_path, sep="\t", index=False)
        print(f"  v2010 total: {len(combined):,} rows written to {output_path}")
        return combined
    else:
        print("No v2010 data extracted.")
        return None


if __name__ == "__main__":
    extract_v2010()

Extracting HCRIS v2010 (2010-2025) from: /scion/5261/econ470001/hcris-data/HCRIS_v2010
  v2010 2010: 2,322 reports
  v2010 2011: 6,143 reports
  v2010 2012: 6,202 reports
  v2010 2013: 6,144 reports
  v2010 2014: 6,190 reports
  v2010 2015: 6,199 reports
  v2010 2016: 6,204 reports
  v2010 2017: 6,121 reports
  v2010 2018: 6,159 reports
  v2010 2019: 6,118 reports
  v2010 2020: 6,050 reports
  v2010 2021: 6,056 reports
  v2010 2022: 6,066 reports
  v2010 2023: 6,103 reports
  v2010 2024: 3,419 reports
  v2010 2025: Files not found (Skipping)
  v2010 total: 85,496 rows written to /home/ikazani/econ470/a0/work/hwk4/data/output/HCRIS_Data_v2010.txt
